##### Регрессия для SI

In [15]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.linear_model import Ridge

In [16]:

# Загрузка подготовленных данных
edata = pd.read_csv('edata_corr.csv')
edata

,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,ExactMolWt,MaxPartialCharge,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiophene,fr_unbrch_alkane,fr_urea,HydrogenMass
0,6.239374,175.482382,28.125000,5.094096,0.387225,0.387225,0.417362,42.928571,384.350449,0.038844,...,0,0,0,0,0,0,0,3,0,44.352
1,0.771831,5.402819,7.000000,3.961417,0.533868,0.533868,0.462473,45.214286,388.381750,0.012887,...,0,0,0,0,0,0,0,3,0,48.384
2,223.808778,161.142320,0.720000,2.627117,0.543231,0.543231,0.260923,42.187500,446.458903,0.094802,...,0,0,0,0,0,0,0,3,0,58.464
3,107.131532,139.270991,1.300000,5.150510,0.270476,0.270476,0.429038,36.514286,466.334799,0.062897,...,0,0,0,0,0,0,0,0,0,42.336
4,15.037911,30.075821,2.000000,5.758408,0.278083,0.278083,0.711012,28.600000,332.225249,0.062897,...,0,0,0,0,0,0,0,0,0,28.224
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729,31.000104,34.999650,1.129017,12.934891,0.048029,-0.476142,0.382752,49.133333,414.240624,0.317890,...,0,0,0,0,0,0,0,0,0,34.272
730,31.999934,33.999415,1.062484,13.635345,0.030329,-0.699355,0.369425,44.542857,485.277738,0.327562,...,0,0,0,0,0,0,0,0,0,39.312
731,30.999883,33.999458,1.096761,13.991690,0.026535,-0.650790,0.284923,41.973684,545.281109,0.327887,...,1,0,0,0,0,0,0,0,0,43.344
732,31.998959,32.999644,1.031272,13.830180,0.146522,-1.408652,0.381559,39.000000,522.282883,0.312509,...,0,0,0,0,0,0,0,0,0,42.336


In [17]:

# Разделение признаков и целевой переменной
X = edata.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = edata['SI']

In [18]:
# Разделим выборку на тестовую и тренировочную
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
# Функция обучения и оценки модели
def eval_fit_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred)**0.5
    r2 = r2_score(y_test, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Функция оценки модели
def eval_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred)**0.5
    r2 = r2_score(y_test, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

Будем используем модели CatBoost, XGBoost и RandomForest, поскольку:
у нас относительно небольшая выборка (менее 1000 объектов),
присутствует много признаков, среди которых возможна мультиколлинеарность,
а также есть признаки с выбросами и неоднородными масштабами.
LinearRegression возьмем для сравнения в качестве простого ориентира.


In [20]:
# Список моделей
models = {
    "CatBoost": CatBoostRegressor(verbose=0, random_state=42),
    "XGBoost": XGBRegressor(verbosity=0, random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "LinearRegression": LinearRegression(),
}


##### Обучим модели с применением базовых настроек

In [21]:
# Обучение и оценка
results = {name: eval_fit_model(model, X_train, X_test, y_train, y_test) for name, model in models.items()}
results_df = pd.DataFrame(results).T
print(results_df)

                       MAE      RMSE        R2
CatBoost          6.274204  8.981574  0.068564
XGBoost           6.650447  9.343823 -0.008085
RandomForest      6.548547  8.866712  0.092235
LinearRegression  6.604684  8.890007  0.087459


Стандартизируем признаки и применим метод главных компонент (PCA) для уменьшения размерности признакового пространства, сохранив при этом 90% дисперсии исходных данных.

In [22]:
# Стандартизация
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA с сохранением 90% дисперсии 
pca = PCA(n_components=0.9)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Преобразуем numpy в DataFrame
pca_columns = [f'pca_{i}' for i in range(X_train_pca.shape[1])]
X_train_pca = pd.DataFrame(X_train_pca, columns=pca_columns, index=X_train.index)
X_test_pca = pd.DataFrame(X_test_pca, columns=pca_columns, index=X_test.index)

In [23]:
# Модели
models = {
    "CatBoost": CatBoostRegressor(verbose=0, random_state=42),
    "XGBoost": XGBRegressor(verbosity=0, random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "LinearRegression": LinearRegression()
}

In [24]:
# Обучение моделей, получение метрик 
results_pca = {
    name: eval_fit_model(model, X_train_pca, X_test_pca, y_train, y_test)
    for name, model in models.items()
}

results_pca_df = pd.DataFrame(results_pca).T
print(results_pca_df)

                       MAE      RMSE        R2
CatBoost          6.425287  9.001168  0.064496
XGBoost           6.509105  9.486439 -0.039093
RandomForest      6.738928  9.092708  0.045371
LinearRegression  6.755980  8.996172  0.065534


PCA отрицательно сказался на всех метриках

Теперь подберем гиперпараметры с помощью GridSearchCV. Вместо LinearRegression будем использовать Ridge, поскольку у LinearRegression отсутствуют гиперпараметры для настройки.

In [25]:
 # GridSearchCV параметры
rf_params = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 3, 5],
}

xgb_params = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 5],
    'booster': ['gbtree'], 
    'tree_method': ['gpu_hist']
}

ridge_params = {
    'alpha': [0.1, 1, 10, 100, 1000]
}

cat_params = {
    'depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'iterations': [100, 300],
    'l2_leaf_reg': [1, 5, 9]
}


# Модели с GridSearch
rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='neg_mean_squared_error'
)

xgb = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    param_grid=xgb_params,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=1,
    verbose=1,
    error_score='raise'
)

ridge = GridSearchCV(
    Ridge(),
    ridge_params,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='neg_mean_squared_error'
)

cat = GridSearchCV(
    CatBoostRegressor(verbose=0, random_state=42),
    param_grid=cat_params,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

In [26]:
# Обучение моделей с подбором параметров
rf.fit(X_train_pca, y_train)
xgb.fit(X_train_pca, y_train)
ridge.fit(X_train_pca, y_train)
cat.fit(X_train_pca, y_train)

# Метрики моделей с лучшими параметрами
results = {
    "Best RandomForest": eval_model(rf.best_estimator_, X_test_pca, y_test),
    "Best XGBoost": eval_model(xgb.best_estimator_, X_test_pca, y_test),
    "Best Ridge": eval_model(ridge.best_estimator_, X_test_pca, y_test),
    "Best CatBoost": eval_model(cat.best_estimator_, X_test_pca, y_test),
}

# Посмотрим лучшие параметры
print("Best Params RF:", rf.best_params_)
print("Best Params XGB:", xgb.best_params_)
print("Best Params Ridge:", ridge.best_params_)
print("Best Params Cat:", cat.best_params_)

# Посмотрим метрики
print("\n Metrics")
print(pd.DataFrame(results).T)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Fitting 3 folds for each of 432 candidates, totalling 1296 fits
Fitting 3 folds for each of 5 candidates, totalling 15 fits
Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best Params RF: {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 200}
Best Params XGB: {'booster': 'gbtree', 'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100, 'reg_alpha': 1, 'reg_lambda': 5, 'subsample': 0.8, 'tree_method': 'gpu_hist'}
Best Params Ridge: {'alpha': 1000}
Best Params Cat: {'depth': 8, 'iterations': 100, 'l2_leaf_reg': 5, 'learning_rate': 0.05}

 Metrics
                        MAE      RMSE        R2
Best RandomForest  6.552808  8.881255  0.089255
Best XGBoost       6.347485  8.763108  0.113325
Best Ridge         6.739673  8.919716  0.081350
Best CatBoost      6.521412  8.864097  0.092771


Подбор гиперпараметров + PCA улучшил XGBoost и незначительно остальные метрики.

Попробуем подобрать гиперпараметры и обучить модели на данных без PCA.

In [27]:
# Обучение моделей с подбором параметров
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)
cat.fit(X_train, y_train)

# Метрики моделей с лучшими параметрами
results = {
    "Best RandomForest": eval_model(rf.best_estimator_, X_test, y_test),
    "Best XGBoost": eval_model(xgb.best_estimator_, X_test, y_test),
    "Best CatBoost": eval_model(cat.best_estimator_, X_test, y_test),
}

# Посмотрим лучшие параметры
print("Best Params RF:", rf.best_params_)
print("Best Params XGB:", xgb.best_params_)
print("Best Params Cat:", cat.best_params_)

# Посмотрим метрики
print("\n Metrics")
print(pd.DataFrame(results).T)

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Fitting 3 folds for each of 432 candidates, totalling 1296 fits
Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best Params RF: {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 150}
Best Params XGB: {'booster': 'gbtree', 'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100, 'reg_alpha': 0, 'reg_lambda': 5, 'subsample': 1.0, 'tree_method': 'gpu_hist'}
Best Params Cat: {'depth': 4, 'iterations': 100, 'l2_leaf_reg': 9, 'learning_rate': 0.1}

 Metrics
                        MAE      RMSE        R2
Best RandomForest  6.366227  8.617044  0.142637
Best XGBoost       6.312919  8.776830  0.110546
Best CatBoost      6.468937  8.659285  0.134211


Удалось улучшить метрики у RandomForest и CatBoost.
Матрица признаков, используемая в модели Ridge, является плохо обусловленной.

#### Оценим полученные метрики в контексте статистик целевых переменных.

In [28]:
# Целевые переменные
target = ['IC50, mM', 'CC50, mM', 'SI']

# Описательная статистика
target_stats = edata[target].describe().T
target_stats['median'] = edata[target].median()

print(target_stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'median']])

          count        mean         std       min        25%         50%  \
IC50, mM  734.0  115.896009  156.219852  0.108830  15.099820   43.049903   
CC50, mM  734.0  384.109715  367.306455  0.700808  79.059629  258.410909   
SI        734.0    7.674797    8.947304  0.011489   1.787829    3.545013   

                 75%          max      median  
IC50, mM  138.147153   705.293226   43.049903  
CC50, mM  623.450022  1536.043255  258.410909  
SI          9.893714    38.168094    3.545013  


Выводы:
- MAE 6.3 - 6.5 почти сравнимо с средним значением SI 7.67 — модели допускают высокую абсолютную ошибку.
- R2 < 0.15 — очень низкое значение, то есть меньше 15% дисперсии SI объясняется моделью.
- RandomForest с настройкой показал наилучшее качество (MAE, RMSE, R²).
- Ни одна модель не предсказывает SI с высокой точностью.

Это связано с тем, что SI = CC50 / IC50 — агрегированная метрика, зависящая от двух источников ошибки.

Рекомендации:
- Не использовать SI как основную цель для регрессии.
- Построить отдельные модели для IC50 и CC50.
- Рассчитать SI как отношение CC50_pred / IC50_pred.